# Phase 3 — Churn Intelligence (XGBoost + Cox + Uplift)

Builds a full churn intelligence stack on top of the Phase 2 CLV scores.

**Sections**
1. Setup & data preparation
2. Step 3.1 — Define churn label
3. Step 3.2 — XGBoost churn classifier
4. Step 3.3 — SHAP explanations
5. Step 3.4 — Cox Proportional Hazards survival model
6. Step 3.5 — Uplift model (T-Learner)
7. Step 3.6 — Merge full intelligence table

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from lifelines import CoxPHFitter, KaplanMeierFitter

from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, build_master_customer_table,
)
from models.churn_model import (
    define_churn_label, prepare_churn_features,
    fit_xgboost_classifier, evaluate_classifier,
    get_shap_values, get_top_shap_drivers,
    fit_cox_model, predict_survival_days,
    compute_treatment_proxy, fit_uplift_tlearner, score_uplift,
    build_full_intelligence_table,
    CHURN_FEATURES,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, df_all = clean_data(df_raw)
master = build_master_customer_table(df_customers, return_features, cancel_features)

clv_scores = pd.read_csv('../data/processed/clv_scores.csv')
print(f'\nmaster shape     : {master.shape}')
print(f'clv_scores shape : {clv_scores.shape}')

## Step 3.1 — Define Churn Label

**Definition:** `churned = 1` if `recency > 180 days`, else `0`.

Recency is the number of days since the customer's last purchase, measured from the dataset snapshot date. Customers who haven't bought in more than 6 months are considered churned.

In [ ]:
master_labelled = define_churn_label(master)

fig, ax = plt.subplots(figsize=(6, 4))
counts = master_labelled['churned'].value_counts().sort_index()
bars = ax.bar(['Active (0)', 'Churned (1)'], counts.values,
              color=['#5cb85c', '#d9534f'], width=0.5)
ax.set_title('Churn Label Distribution', fontsize=13)
ax.set_ylabel('Customer Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + val * 0.01,
            f'{val:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## Step 3.2 — XGBoost Churn Classifier

**Features used:** recency, frequency, monetary, velocity_decay_ratio, category_hhi, spend_cv, return_rate, cancellation_rate, customer_tenure_days, avg_items_per_order, country_count

**Class imbalance handling:** `scale_pos_weight = count(churned=0) / count(churned=1)` — no SMOTE.

**Business question:** Which customers are most likely to churn in the next 180 days, and how well can we predict it?

In [ ]:
X, y = prepare_churn_features(master_labelled)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'train size : {len(X_train):,}  (churned: {y_train.sum():,})')
print(f'test size  : {len(X_test):,}   (churned: {y_test.sum():,})')

In [ ]:
xgb_model = fit_xgboost_classifier(X_train, y_train, X_test, y_test)

---
### Evaluation — Confusion Matrix & ROC Curve
**Business question:** How precisely can we identify churners, and how does the classifier trade off false positives against false negatives?

In [ ]:
metrics = evaluate_classifier(xgb_model, X_test, y_test)
print(f"\nAUC-ROC   : {metrics['auc']:.4f}")
print(f"Precision : {metrics['precision']:.4f}")
print(f"Recall    : {metrics['recall']:.4f}")
print(f"F1        : {metrics['f1']:.4f}")

In [ ]:
import joblib
from pathlib import Path
Path('../models').mkdir(parents=True, exist_ok=True)
joblib.dump(xgb_model, '../models/churn_xgb.pkl')

churn_scores = master_labelled[['customer_id']].copy()
churn_scores['churn_probability'] = xgb_model.predict_proba(X)[:, 1]
print('saved models/churn_xgb.pkl')
print(churn_scores['churn_probability'].describe().round(4))

## Step 3.3 — SHAP Explanations

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions, giving us **global** feature importance and **local** explanations for individual customers.

**Business question:** Which behavioural signals drive churn predictions — and why is a specific high-risk customer flagged?

In [ ]:
explainer, shap_values = get_shap_values(xgb_model, X_test)

# Plot 1 — SHAP beeswarm (dot) summary
shap.summary_plot(shap_values, X_test, show=True)

In [ ]:
# Plot 2 — SHAP bar chart (mean absolute SHAP value per feature)
shap.summary_plot(shap_values, X_test, plot_type='bar', show=True)

In [ ]:
# Plot 3 — SHAP waterfall for one at-risk customer (churn_prob > 0.7)
y_prob_test = metrics['y_prob']
at_risk_idx = np.where(y_prob_test > 0.7)[0]
print(f'customers with churn_prob > 0.7 in test set: {len(at_risk_idx):,}')

if len(at_risk_idx) > 0:
    idx = at_risk_idx[0]
    shap_exp = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_test.iloc[idx].values,
        feature_names=CHURN_FEATURES,
    )
    shap.plots.waterfall(shap_exp, show=True)

# save shap values for use in API
np.save('../data/processed/shap_values.npy', shap_values)
print('saved data/processed/shap_values.npy')

## Step 3.4 — Cox Proportional Hazards Survival Model

The Cox model treats **customer_tenure_days** as the duration and **churned** as the event indicator. Customers with `churned=0` are **right-censored** — they are still active at the time of observation, so we know they survived *at least* that long but do not know their eventual churn time.

Hazard ratios > 1 indicate a feature increases churn risk; < 1 indicates a protective effect.

**Business question:** Which customer attributes accelerate time-to-churn, and when (in days) do we expect a given customer to churn?

In [ ]:
cox_features = ['recency', 'frequency', 'monetary', 'velocity_decay_ratio',
                'category_hhi', 'spend_cv', 'return_rate', 'cancellation_rate',
                'avg_items_per_order', 'country_count', 'customer_tenure_days', 'churned']

df_cox = master_labelled[cox_features].fillna(0).copy()
# Cox requires duration > 0
df_cox['customer_tenure_days'] = df_cox['customer_tenure_days'].clip(lower=1)

cph = fit_cox_model(df_cox)
cph.print_summary()

---
### Plot 1 — Hazard Ratio Forest Plot
**Business question:** Which features most strongly increase or decrease churn hazard, and are those effects statistically significant?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
cph.plot(ax=ax)
ax.set_title('Cox PH — Hazard Ratio Forest Plot (95% CI)')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

joblib.dump(cph, '../models/cox_model.pkl')
print('saved models/cox_model.pkl')